In [ ]:
from google.colab import files
uploaded = files.upload()

Saving heart.csv to heart.csv


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA


df = pd.read_csv("heart.csv")

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

categorical_cols = X.select_dtypes(include=['object', 'category']).columns

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def evaluate_models(X_train, X_test, y_train, y_test, scenario_name=""):
    print(f"--- Model Performance {scenario_name} ---")

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Support Vector Machine (SVM)": SVC(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42)
    }

    accuracies = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        accuracies[name] = acc
        print(f"{name}: {acc:.4f}")

    return accuracies

print("Evaluating models on Original Features (Scaled):")
original_accuracies = evaluate_models(X_train_scaled, X_test_scaled, y_train, y_test, "(Original)")
print("\n")

pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original number of features: {X_train_scaled.shape[1]}")
print(f"Number of features after PCA: {X_train_pca.shape[1]}\n")

print("Evaluating models on PCA-Reduced Features:")
pca_accuracies = evaluate_models(X_train_pca, X_test_pca, y_train, y_test, "(PCA)")
print("\n")

print("--- Accuracy Comparison (Original vs PCA) ---")
for model_name in original_accuracies.keys():
    orig_acc = original_accuracies[model_name]
    pca_acc = pca_accuracies[model_name]
    diff = pca_acc - orig_acc
    print(f"{model_name}:")
    print(f"  Original: {orig_acc:.4f} | PCA: {pca_acc:.4f} | Difference: {diff:+.4f}")

Evaluating models on Original Features (Scaled):
--- Model Performance (Original) ---
Logistic Regression: 0.8533
Support Vector Machine (SVM): 0.8750
Random Forest: 0.8750


Original number of features: 15
Number of features after PCA: 13

Evaluating models on PCA-Reduced Features:
--- Model Performance (PCA) ---
Logistic Regression: 0.8533
Support Vector Machine (SVM): 0.8804
Random Forest: 0.8641


--- Accuracy Comparison (Original vs PCA) ---
Logistic Regression:
  Original: 0.8533 | PCA: 0.8533 | Difference: +0.0000
Support Vector Machine (SVM):
  Original: 0.8750 | PCA: 0.8804 | Difference: +0.0054
Random Forest:
  Original: 0.8750 | PCA: 0.8641 | Difference: -0.0109
